# Forecasting the mid-price

Section 1.4 of the lecture notes, from the model of the order flow to the forecast of the
mid-price.  Sections 1 to 5 read the model: the two specifications the package ships, their
symmetry, their baselines, the intensity contrast and the signed excitation.  Sections 6 to 12
take the statements of section 1.4.3 one at a time, and set each against a Monte Carlo or
against a second computation that shares none of its arithmetic.

The Monte Carlo cells of sections 6, 7, 8 and 11 take tens of seconds each, and every one is
seeded.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from unito26.lob import config
from unito26.lob.hawkes import (
    ExponentialHawkes, HawkesParams, decayed_counts_at, intensities_at_events,
)
from unito26.lob.idealised import AssumptionBreach, IdealisedFlow, UniformDepth, idealised_book
from unito26.lob.messages import BUY, SELL, BranchingRatio, GridDepth, Horizon
from unito26.lob.simulate import EventJournal, EventType
from unito26.lob.visualization import INK, MUTED, PALETTE, monte_carlo_figure, use_template

use_template()

LABELS = ["market buy", "market sell", "limit buy", "limit sell", "withdraw bid", "withdraw ask"]
PAIRS = {"market": EventType.MARKET_BUY, "limit": EventType.LIMIT_BUY,
         "withdrawal": EventType.WITHDRAW_BUY}
PRESSURE = np.array([event.pressure for event in EventType], dtype=float)
UNIT = np.eye(len(EventType))
IDENTITY = np.eye(len(EventType))

REGIMES = {
    "example": config.example_order_flow_params(),
    "trending": config.trending_order_flow_params(),
}
MARKS = {
    "example": config.example_mark_params(),
    "trending": config.trending_mark_params(),
}
COLOUR = {"example": PALETTE[0], "trending": PALETTE[1]}
DASH = {"market": "solid", "limit": "dash", "withdrawal": "dot"}

## 1. Six types, and two regimes

The order flow of Definition 1.4.6 has six types, and $\varpi$ records the pressure each exerts
on the mid-price: $+1$ for the three that push it up, $-1$ for their mirrors.

In [ ]:
print(pd.Series(PRESSURE.astype(int), index=LABELS, name="pressure").to_frame().T.to_string())

The package ships two specifications.  The column sums of $\Gamma$ count the direct offspring
of one event of each type (Proposition 1.3.8).  The signed endogenous fraction counts the
direct offspring that carry their parent's pressure, net of those that carry the opposite one,
per event at the stationary rates.

In [ ]:
print(pd.concat({
    name: pd.DataFrame({
        "stationary intensity": params.stationary_intensity(),
        "direct offspring": params.branching_matrix.sum(axis=0),
    }, index=LABELS)
    for name, params in REGIMES.items()
}, axis=1).round(3).to_string())
print()
print(pd.DataFrame({
    name: {
        "branching ratio": params.branching_ratio,
        "total rate (1/s)": params.stationary_intensity().sum(),
        "signed endogenous fraction": params.signed_endogenous_fraction(PRESSURE),
    }
    for name, params in REGIMES.items()
}).round(3).to_string())

The two specifications share the branching ratio and the total rate, and their stationary
intensities are close type by type.  They differ in how the offspring are spread: in the
example a market order begets the most and a limit order the least, while in the trending
specification the offspring counts are closer together.  And the signed endogenous fraction
has opposite signs.  On balance, the direct offspring of an event push against it in the
example and with it in the trending specification.

## 2. Direction symmetry

$\Sigma$ exchanges the two members of each pair, so that $\Sigma\varpi = -\varpi$.
Definition 1.4.9 asks $\Sigma A \Sigma = A$ and $\Sigma\mu = \mu$, and Proposition 1.4.10 then
gives $\varpi^\top\mu = 0$ and, at $\rho < 1$, $\varpi^\top\lambda^* = 0$.

In [ ]:
SWAP = UNIT[[1, 0, 3, 2, 5, 4]]

print("Sigma varpi = -varpi:", np.array_equal(SWAP @ PRESSURE, -PRESSURE))
print()
print(pd.DataFrame({
    name: {
        "max |Sigma A Sigma - A|": np.abs(SWAP @ params.excitation @ SWAP - params.excitation).max(),
        "max |Sigma mu - mu|": np.abs(SWAP @ params.baseline - params.baseline).max(),
        "varpi' mu": PRESSURE @ params.baseline,
        "varpi' lambda*": PRESSURE @ params.stationary_intensity(),
    }
    for name, params in REGIMES.items()
}).to_string(float_format="{:.1e}".format))

Both specifications are direction-symmetric, and both consequences hold to rounding.

The hypothesis bears on the kernel as well as on the baseline, and $\varpi^\top\mu = 0$ cannot
stand in for it.  Below, one entry of the example's kernel is halved and its mirror image left
alone: market sells now beget fewer limit buys than market buys beget limit sells.  The
baseline is untouched, so $\varpi^\top\mu$ is still zero, and the stationary intensity is no
longer balanced.

In [ ]:
example = REGIMES["example"]
lopsided = example.excitation.copy()
lopsided[EventType.LIMIT_BUY, EventType.MARKET_SELL] /= 2
broken = HawkesParams(baseline=example.baseline, excitation=lopsided, decay=example.decay)

print("max |Sigma A Sigma - A|:", np.abs(SWAP @ lopsided @ SWAP - lopsided).max())
print("varpi' mu              :", PRESSURE @ broken.baseline)
print("varpi' lambda*         :", round(PRESSURE @ broken.stationary_intensity(), 4))

Reordered so that the three types pushing up come first and their mirrors second, $\Gamma$
splits into four blocks.  $\Sigma$ exchanges the two diagonal blocks, and the two off-diagonal
ones, so under symmetry each pair of blocks is equal.  What distinguishes the regimes is where
the mass sits: the example puts it across the partition, where what depletes a side begets what
refills it, and the trending specification puts it within.

In [ ]:
ORDER = [EventType.MARKET_BUY, EventType.LIMIT_BUY, EventType.WITHDRAW_SELL,
         EventType.MARKET_SELL, EventType.LIMIT_SELL, EventType.WITHDRAW_BUY]
across = np.not_equal.outer(PRESSURE, PRESSURE)
for name, params in REGIMES.items():
    gamma = params.branching_matrix
    print(f"{name:<9} share of the mass of Gamma across the partition: "
          f"{gamma[across].sum() / gamma.sum():.3f}")

top = max(params.branching_matrix.max() for params in REGIMES.values())
figure = make_subplots(rows=1, cols=2, subplot_titles=list(REGIMES), horizontal_spacing=0.2)
for column, params in enumerate(REGIMES.values(), start=1):
    names = [LABELS[event] for event in ORDER]
    figure.add_trace(go.Heatmap(
        z=params.branching_matrix[np.ix_(ORDER, ORDER)], x=names, y=names,
        zmin=0, zmax=top, colorscale="Blues", showscale=column == 2,
        texttemplate="%{z:.2f}", hovertemplate="%{x} begets %{y}: %{z:.3f}<extra></extra>",
    ), row=1, col=column)
    figure.add_vline(x=2.5, line=dict(color=INK, width=1.5), row=1, col=column)
    figure.add_hline(y=2.5, line=dict(color=INK, width=1.5), row=1, col=column)
figure.update_yaxes(autorange="reversed")
figure.update_xaxes(title_text="exciting type")
figure.update_layout(height=500, title="Gamma in pressure blocks, up types first: the column type begets the row type")
figure

The proof of Proposition 1.4.10 uses nothing about $M$ beyond $\Sigma M \Sigma = M$, and the
matrices the forecast is built from all qualify.  Each is checked below for the invariance and
for $\varpi^\top M\mu = 0$, at a horizon of one second.

In [ ]:
rows = []
for name, params in REGIMES.items():
    for label, matrix in {
        "exp(K h)": params.mean_response(Horizon(1.0)),
        "Phi(h)": params.mean_response_integral(Horizon(1.0)),
        "(I - Gamma)^-1": np.linalg.inv(IDENTITY - params.branching_matrix),
    }.items():
        rows.append({
            "regime": name,
            "M": label,
            "max |Sigma M Sigma - M|": np.abs(SWAP @ matrix @ SWAP - matrix).max(),
            "varpi' M mu": PRESSURE @ matrix @ params.baseline,
        })
print(pd.DataFrame(rows).to_string(index=False, float_format="{:.1e}".format))

## 3. Choosing a specification

Equation (1.4.8) reads the baseline off the excitation structure and the stationary intensity,
$\mu = (I - \Gamma)\lambda^*$.  The shipped baselines are recovered from their own stationary
intensities.

In [ ]:
for name, params in REGIMES.items():
    recovered = (IDENTITY - params.branching_matrix) @ params.stationary_intensity()
    print(f"{name:<9} max |(I - Gamma) lambda* - mu| = "
          f"{np.abs(recovered - params.baseline).max():.1e}")

The pair $(\Gamma, \lambda^*)$ is admissible when this $\mu$ is strictly positive.  Type by
type, $\mu_e/\lambda^*_e$ is the share of the events of type $e$ that arrive as immigrants, and
the type with the smallest share binds first.  Scaling the kernel at a fixed $\lambda^*$ scales
every $(\Gamma\lambda^*)_e$ with $\rho$, so the largest branching ratio at which the same
target stays reachable is $\rho\,\min_e \lambda^*_e/(\Gamma\lambda^*)_e$.

In [ ]:
print(pd.DataFrame({
    name: params.baseline / params.stationary_intensity() for name, params in REGIMES.items()
}, index=LABELS).round(3).to_string())
print()

LIMIT = {}
for name, params in REGIMES.items():
    target = params.stationary_intensity()
    LIMIT[name] = params.branching_ratio * (target / (params.branching_matrix @ target)).min()
    first = int(np.argmin(params.baseline / target))
    mirror = int(np.flatnonzero(SWAP[first])[0])
    print(f"{name:<9} binding: {LABELS[first]} and {LABELS[mirror]}; largest branching "
          f"ratio for this target: {LIMIT[name]:.4f}")

The binding type differs between the regimes.  In the example it is the limit orders, whose
rate is almost all offspring, most of it of the types that deplete their side of the book; in
the trending specification it is the market orders.  Past the limit the constructor refuses the
target, and names the components that went negative.

In [ ]:
for name, params in REGIMES.items():
    target = params.stationary_intensity()
    below = HawkesParams.from_stationary_intensity(
        target, params.excitation, params.decay, BranchingRatio(LIMIT[name] - 0.01))
    print(f"{name}, rho = {LIMIT[name] - 0.01:.4f}: accepted, smallest baseline "
          f"{below.baseline.min():.4f}")
    try:
        HawkesParams.from_stationary_intensity(
            target, params.excitation, params.decay, BranchingRatio(LIMIT[name] + 0.01))
    except ValueError as refusal:
        print(f"{name}, rho = {LIMIT[name] + 0.01:.4f}: {refusal}")
    print()

## 4. The intensity contrast along a path

Definition 1.4.11 and equation (1.4.10): under direction symmetry the contrast
$\Delta\lambda = \lambda^{\uparrow} - \lambda^{\downarrow} = \varpi^\top\lambda$ is
$\varpi^\top A Z$, a function of the state.  We simulate one path of the example specification
from the empty state, and read it over its last two seconds, long after the start has been
forgotten.

In [ ]:
PATH = REGIMES["example"]
START, END = 10.0, 12.0
events = list(ExponentialHawkes(PATH, np.random.default_rng(4)).events(END))
times = np.array([t for t, _ in events])
types = np.array([event for _, event in events])
shown = times > START
contrast_row = PRESSURE @ PATH.excitation

clock = np.linspace(START, END, 4001)
states_on_clock = decayed_counts_at(PATH.decay, times, types, np.zeros(6), clock)
before = decayed_counts_at(PATH.decay, times, types, np.zeros(6), times) @ contrast_row
after = before + contrast_row[types]

print(f"{shown.sum()} events in ({START}, {END}] s, {times.size} since the start")
print("max |varpi' lambda(T-) - varpi' A Z(T-)| over every event:",
      np.abs(intensities_at_events(PATH, times, types) @ PRESSURE - before).max())
print()
print(pd.DataFrame({
    "time (s)": times[shown],
    "type": [LABELS[event] for event in types[shown]],
    "contrast at Z(T-)": before[shown],
    "contrast at Z(T+)": after[shown],
}).head(8).round(3).to_string(index=False))

The two pre-jump readings agree.  `intensities_at_events` reads $\varpi^\top\lambda(T_n-)$,
which carries the baseline term $\varpi^\top\mu = 0$; `decayed_counts_at` on the event times
reads $\varpi^\top A Z(T_n-)$, because $Z$ is left-continuous.  The forecast of section 6 needs
the other value, $Z(T_n+) = Z(T_n-) + \mathbf e_{E_n}$, the state that includes the event just
observed.  Each event moves the contrast by $(\varpi^\top A)_{E_n}$.  Wherever this notebook
forecasts from an event, it uses $Z(T_n+)$.

In [ ]:
figure = go.Figure([
    go.Scatter(x=clock, y=states_on_clock @ contrast_row, mode="lines", name="on a clock",
               line=dict(color=MUTED, width=1.5)),
    go.Scatter(x=times[shown], y=before[shown], mode="markers", name="at an event, from Z(T-)",
               marker=dict(color=INK, size=8, symbol="circle-open")),
    go.Scatter(x=times[shown], y=after[shown], mode="markers", name="at an event, from Z(T+)",
               marker=dict(color=COLOUR["example"], size=8)),
])
figure.update_xaxes(title_text="time (s)")
figure.update_yaxes(title_text="intensity contrast (1/s)")
figure.update_layout(title="The intensity contrast along a path of the example specification",
                     height=420, legend=dict(orientation="h", y=1.1, x=0))
figure

## 5. Signed excitation, pair by pair

Proposition 1.4.12: $\theta_e = (\varpi^\top A)_e/\varpi_e$ takes one value on each pair.
$\theta_e/\beta$ counts the direct offspring of one event of type $e$ that push its way, net of
those that push the other way.

In [ ]:
theta = pd.DataFrame({
    name: params.signed_excitation(PRESSURE) for name, params in REGIMES.items()
}, index=LABELS)
for name in REGIMES:
    values = theta[name].to_numpy()
    assert np.allclose(values[0::2], values[1::2])
print(theta.iloc[0::2].set_axis(list(PAIRS)).round(3).to_string())
print()
for name, params in REGIMES.items():
    weights = params.stationary_intensity() / params.stationary_intensity().sum()
    print(f"{name:<9} sum_e (lambda*_e / lambdabar) theta_e / beta = "
          f"{weights @ params.signed_excitation(PRESSURE) / params.decay:.4f}   "
          f"signed endogenous fraction = {params.signed_endogenous_fraction(PRESSURE):.4f}")

In the example $\theta$ is negative on the market and withdrawal pairs and zero on the limit
pair.  A market buy, or a withdrawal from the ask, begets more of the flow that pushes down than
of the flow that pushes up, and a limit order begets neither on balance.  In the trending
specification $\theta$ is positive on every pair.  Weighted by the stationary rates,
$\theta/\beta$ is the signed endogenous fraction of section 1.

Equation (1.4.12) reassembles the contrast of section 4 from $\theta$ and the state.

In [ ]:
reassembled = states_on_clock @ (PRESSURE * PATH.signed_excitation(PRESSURE))
print("max |sum_e varpi_e theta_e Z_e - varpi' A Z| on the clock:",
      np.abs(reassembled - states_on_clock @ contrast_row).max())

## 6. The short-horizon forecast against Monte Carlo

Proposition 1.4.13: from the state $Z(t+) = z$, the expected pressure-signed size over
$(t, t+h]$ is $\bar q\,\varpi^\top A\Phi(h)z$.  We fix one state, the one just after a market
buy that arrived on the mean state $m = \lambda^*/\beta$ of Proposition 1.3.20:
$$z = m + \mathbf e_{\mathrm{MB}}.$$
Its baseline part is annihilated, $\varpi^\top A\Phi(h)m = 0$ by Proposition 1.4.10, so the
forecast is $\bar q\,\theta_{\mathrm{MB}}(h)$, with $\theta(h)$ of (1.4.16).

In [ ]:
HORIZONS = np.array([0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 3.0])
MEAN_STATE = {name: params.stationary_intensity() / params.decay for name, params in REGIMES.items()}
MARKET_BUY, MARKET_SELL = UNIT[EventType.MARKET_BUY], UNIT[EventType.MARKET_SELL]


def flow_forecast(params, state, horizons):
    '''varpi' A Phi(h) z at every horizon: Proposition 1.4.13 per unit of mean size.'''
    return np.array([
        PRESSURE @ params.excitation @ params.mean_response_integral(Horizon(h)) @ state
        for h in horizons
    ])


for name, params in REGIMES.items():
    print(f"{name:<9} max over the horizons of |varpi' A Phi(h) m| = "
          f"{np.abs(flow_forecast(params, MEAN_STATE[name], HORIZONS)).max():.1e}")

The marks are the sizes of `MarkParams`: lognormal, rounded to the lot $\ell$, at least one lot.
So $q = \ell K$ with $K = \max(1, \operatorname{round}(X/\ell))$ and $X$ lognormal, and the
mean is a sum over lots, $\bar q = \ell\sum_{k \ge 1} k\,\mathbb P(K = k)$.  The two mark
specifications share their size distribution.

In [ ]:
def draw_sizes(marks, rng, count):
    raw = rng.lognormal(marks.mean_log_size, marks.sigma_log_size, count)
    return np.maximum(marks.lot, np.round(raw / marks.lot).astype(int) * marks.lot)


def mean_size(marks):
    lots = np.arange(1, 10_000)
    below = stats.norm.cdf(
        (np.log((lots + 0.5) * marks.lot) - marks.mean_log_size) / marks.sigma_log_size)
    return marks.lot * float(lots @ np.diff(np.r_[0.0, below]))


MEAN_SIZE = {name: mean_size(marks) for name, marks in MARKS.items()}
for name, marks in MARKS.items():
    sample = draw_sizes(marks, np.random.default_rng(6), 1_000_000)
    print(f"{name:<9} qbar summed over lots = {MEAN_SIZE[name]:.3f}   "
          f"mean of a million draws = {sample.mean():.3f} +- {sample.std() / 1000:.3f}")

Each path starts afresh from $Z(0+) = z$ and $N(0) = 0$ (`ExponentialHawkes.from_state`),
runs to the last horizon and is read at every horizon; the sizes are drawn independently of the
path.  With unit sizes the same sum counts signed events, and $\bar q = 1$.

In [ ]:
PATHS = 20_000


def signed_flow(params, marks, state, paths, rng):
    '''sum of varpi_{E_n} q_n over (0, h] for every horizon, one row per path: with unit
    sizes, and with sizes drawn from marks.'''
    unit = np.empty((paths, HORIZONS.size))
    sized = np.empty((paths, HORIZONS.size))
    for path in range(paths):
        events = list(ExponentialHawkes.from_state(params, state, rng).events(HORIZONS[-1]))
        arrivals = np.array([t for t, _ in events], dtype=float)
        signs = PRESSURE[np.array([event for _, event in events], dtype=int)]
        sizes = draw_sizes(marks, rng, signs.size)
        read = np.searchsorted(arrivals, HORIZONS, side="right")
        unit[path] = np.r_[0.0, np.cumsum(signs)][read]
        sized[path] = np.r_[0.0, np.cumsum(signs * sizes)][read]
    return unit, sized


def against(samples, formula, horizons):
    mean = samples.mean(axis=0)
    error = samples.std(axis=0, ddof=1) / np.sqrt(samples.shape[0])
    return pd.DataFrame({"formula": formula, "Monte Carlo": mean, "standard error": error,
                         "gap / SE": (mean - formula) / error}, index=pd.Index(horizons, name="h (s)"))


FANS = {
    name: signed_flow(params, MARKS[name], MEAN_STATE[name] + MARKET_BUY, PATHS,
                      np.random.default_rng(61))
    for name, params in REGIMES.items()
}
FORECAST = {name: flow_forecast(params, MEAN_STATE[name] + MARKET_BUY, HORIZONS)
            for name, params in REGIMES.items()}

for label, slot, scale in (("sizes of the mark specification", 1, MEAN_SIZE),
                           ("unit sizes", 0, {name: 1.0 for name in REGIMES})):
    print(label)
    print(pd.concat({
        name: against(FANS[name][slot], scale[name] * FORECAST[name], HORIZONS) for name in REGIMES
    }, axis=1).round(3).to_string())
    print()

In [ ]:
monte_carlo_figure(
    HORIZONS,
    np.stack([FANS[name][1] for name in REGIMES], axis=2),
    np.stack([MEAN_SIZE[name] * FORECAST[name] for name in REGIMES], axis=1),
    list(REGIMES),
    "From m + e_MB: pressure-signed size over (t, t+h], mean and 2 SE against the forecast",
).update_xaxes(title_text="horizon h (s)").update_yaxes(title_text="shares")

In [ ]:
monte_carlo_figure(
    HORIZONS,
    np.stack([FANS[name][0] for name in REGIMES], axis=2),
    np.stack([FORECAST[name] for name in REGIMES], axis=1),
    list(REGIMES),
    "The same paths with unit sizes: the forecast is theta_MB(h)",
).update_xaxes(title_text="horizon h (s)").update_yaxes(title_text="signed events")

The formula runs through the Monte Carlo in both regimes, with sizes and without, and the gaps
are of the size sampling error produces: a gap beyond two standard errors turns up about once
in twenty points.  The forecast is negative in the example and positive in the trending
specification, from the state $m + \mathbf e_{\mathrm{MB}}$ of each: after a market buy, the
example forecasts reversal of the flow and the trending specification continuation.  With sizes
the forecast carries the factor $\bar q$, which unit sizes hide.

Which state is read at the event matters.  Read before the market buy's own jump, the state is
$m$, and the forecast from it is zero at every horizon.  The Monte Carlo sits many standard errors
away from zero, so it tells the two readings apart.

In [ ]:
for name in REGIMES:
    unit = FANS[name][0]
    distance = unit.mean(axis=0) / (unit.std(axis=0, ddof=1) / np.sqrt(PATHS))
    print(f"{name:<9} distance from the forecast at Z(t-) = m, in standard errors:",
          distance.round(1))

## 7. The forecast of the mid-price, on an idealised book

Corollary 1.4.14 adds the residual to the forecast of the flow:
$\mathbb E[\Delta P^m] = \frac{\tau\bar q}{2\bar S}\varpi^\top A\Phi(h)z + \mathbb E[\varepsilon]$.
We continue the state $z$ of section 6 into a book that satisfies Assumption 1.4.1, stated in
the cell below: unit sizes, so $\bar q = 1$; a queue depth $\bar S$ behind each touch; both
touches holding the same size; a spread wide enough that no path closes it.  Prices are in
ticks, so $\tau = 1$.  `IdealisedFlow` turns each event into its order, `AggregateBook` folds
it, and each path records the mid-price and the two touch sizes after every message.

In [ ]:
DEPTH = UniformDepth(3)
TOUCHES = {BUY: (1000, 2), SELL: (1041, 2)}
LEVELS = GridDepth(100)
BOOK_HORIZONS = np.array([0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 3.0])
BOOK_PATHS = 10_000


def continuations(params, state, depth, touches, paths, rng):
    '''Idealised continuations from Z(0+) = state, read at every horizon: the mid-price
    change, the imbalance, the changes of the two touch sizes, and whether a quote has moved.'''
    shape = (paths, BOOK_HORIZONS.size)
    run = {key: np.empty(shape) for key in ("mid", "imbalance", "bid", "ask", "moved")}
    breaches = exhausted = 0
    for path in range(paths):
        book = idealised_book(depth, touches, LEVELS)
        quotes = (book.best_bid_price, book.best_ask_price)
        journal = EventJournal.empty()
        flow = IdealisedFlow(ExponentialHawkes.from_state(params, state, rng), depth)
        history = [(0.0, book.mid_price, book.best_bid_size, book.best_ask_size, False)]
        moved = False
        try:
            for message in flow.stream(book, BOOK_HORIZONS[-1], journal):
                book.apply(message, record=False)
                moved = moved or (book.best_bid_price, book.best_ask_price) != quotes
                history.append((message.time, book.mid_price, book.best_bid_size,
                                book.best_ask_size, moved))
        except AssumptionBreach:
            breaches += 1
        exhausted += book.best_bid_price is None or book.best_ask_price is None
        history = np.array(history, dtype=float)
        rows = np.searchsorted(history[:, 0], BOOK_HORIZONS, side="right") - 1
        signs = PRESSURE[np.array(journal.types, dtype=int)]
        run["imbalance"][path] = np.r_[0.0, np.cumsum(signs)][rows]
        for column, key in enumerate(("mid", "bid", "ask", "moved"), start=1):
            run[key][path] = history[rows, column] - history[0, column]
    run["moved"] = run["moved"].astype(bool)
    run["residual"] = run["mid"] - run["imbalance"] / (2 * depth)
    run["breaches"], run["exhausted"] = breaches, exhausted
    return run


def check(run, depth):
    identity = 2 * depth * run["mid"] - (run["imbalance"] - run["bid"] + run["ask"])
    explicit = -(run["bid"] - run["ask"]) / (2 * depth)
    print(f"  paths {run['mid'].shape[0]}, assumption breaches {run['breaches']}, "
          f"exhausted sides {run['exhausted']}")
    print(f"  2 Sbar dP - (OFI - dS^b + dS^a) = 0 on every path and horizon: "
          f"{np.array_equal(identity, np.zeros_like(identity))}")
    print(f"  max |eps - (1.4.4)| = {np.abs(run['residual'] - explicit).max():.1e}   "
          f"max |eps| = {np.abs(run['residual']).max():.3f} ticks")
    assert run["breaches"] == 0 and run["exhausted"] == 0


BOOK = {}
for name, params in REGIMES.items():
    BOOK[name] = continuations(params, MEAN_STATE[name] + MARKET_BUY, DEPTH, TOUCHES,
                               BOOK_PATHS, np.random.default_rng(71))
    print(name)
    check(BOOK[name], DEPTH)

No path breached the assumption or emptied a side, so none is discarded, and the identity of
Proposition 1.4.3 holds on every path and at every horizon: in integers, and against the
explicit residual (1.4.4) to rounding.

Each path gives three numbers at each horizon: the mid-price change $\Delta P^m$, the imbalance
$\mathrm{OFI} = \sum\varpi_{E_n}$ and $\varepsilon = \Delta P^m - \mathrm{OFI}/(2\bar S)$.

In [ ]:
def book_forecast(name, state, depth):
    return flow_forecast(REGIMES[name], state, BOOK_HORIZONS) / (2 * depth)


def book_table(name, run, state, depth):
    flow_term = book_forecast(name, state, depth)
    table = against(run["imbalance"] / (2 * depth), flow_term, BOOK_HORIZONS)
    table.columns = ["flow term", "E[OFI]/(2 Sbar)", "standard error", "gap / SE"]
    table["E[dP]"] = run["mid"].mean(axis=0)
    table["E[eps]"] = run["residual"].mean(axis=0)
    table["SE of E[eps]"] = run["residual"].std(axis=0, ddof=1) / np.sqrt(len(run["residual"]))
    return table


def residual_figure(runs, state_of, depth, title):
    figure = make_subplots(rows=1, cols=len(runs), subplot_titles=list(runs))
    for column, (name, run) in enumerate(runs.items(), start=1):
        figure.add_trace(go.Scatter(
            x=BOOK_HORIZONS, y=book_forecast(name, state_of[name], depth), mode="lines",
            name="flow term, formula", line=dict(color=INK, width=2, dash="dash"),
            legendgroup="flow", showlegend=column == 1,
        ), row=1, col=column)
        for label, key, colour in (("E[ΔPᵐ], Monte Carlo", "mid", PALETTE[2]),
                                   ("E[ε], Monte Carlo", "residual", PALETTE[3])):
            samples = run[key]
            figure.add_trace(go.Scatter(
                x=BOOK_HORIZONS, y=samples.mean(axis=0), mode="lines+markers", name=label,
                error_y=dict(array=2 * samples.std(axis=0, ddof=1) / np.sqrt(samples.shape[0]),
                             thickness=1.5, width=3),
                line=dict(color=colour, width=2), marker=dict(size=8),
                legendgroup=key, showlegend=column == 1,
            ), row=1, col=column)
    figure.update_xaxes(type="log", title_text="horizon h (s)")
    figure.update_yaxes(title_text="ticks", col=1)
    figure.update_layout(title=title, height=460, margin=dict(t=140),
                         legend=dict(orientation="h", yanchor="bottom", y=1.12, x=0))
    return figure


STATE = {name: MEAN_STATE[name] + MARKET_BUY for name in REGIMES}
for name in REGIMES:
    print(name)
    print(book_table(name, BOOK[name], STATE[name], DEPTH).round(4).to_string())
    print()

In [ ]:
residual_figure(BOOK, STATE, DEPTH, f"From m + e_MB, Sbar = {DEPTH}: the three terms of (1.4.14)")

$\mathbb E[\mathrm{OFI}]/(2\bar S)$ meets the flow term at every horizon, which is
Proposition 1.4.13 once more, now through the book.  The residual's expectation has the sign
opposite to the flow term over the shortest horizons, where the expected change of the
mid-price trails the expected flow, and is small beside the flow term after that.

The residual is nonetheless far from independent of the flow, which is the first reason
Remark 1.4.15 gives for carrying it.  On a path on which no quote has moved, the flow went into
the queues: $\Delta P^m = 0$, $\mathrm{OFI} = \Delta S^b - \Delta S^a$ and
$\varepsilon = -\mathrm{OFI}/(2\bar S)$ exactly (Remark 1.4.5).  The table gives the share of
such paths and the correlation of $\varepsilon$ with the imbalance.

In [ ]:
def dependence(run, depth):
    still = ~run["moved"]
    queues = run["bid"] - run["ask"]
    assert np.array_equal(run["mid"][still], np.zeros(still.sum()))
    assert np.array_equal(run["imbalance"][still], queues[still])
    return pd.DataFrame({
        "no quote moved": still.mean(axis=0),
        "corr(eps, OFI)": [np.corrcoef(run["residual"][:, j], run["imbalance"][:, j])[0, 1]
                           for j in range(BOOK_HORIZONS.size)],
    }, index=pd.Index(BOOK_HORIZONS, name="h (s)"))


print(pd.concat({name: dependence(BOOK[name], DEPTH) for name in REGIMES}, axis=1)
      .round(3).to_string())

On most paths no quote moves over the first few tens of milliseconds, and there
$\varepsilon = -\mathrm{OFI}/(2\bar S)$ path by path.  The correlation fades as the quotes start
to move, and the residual's expectation fades with it.

Why the expectation fades so soon is an observation of this notebook, not a statement of the
notes.  A touch holds between one and $\bar S$ units, and the flow at one side moves the queue
position of that side by one unit per event, so the touch size is the queue position modulo
$\bar S$.  Once the flow at a side has spread over more than $\bar S$ units, the touch size is
close to uniform on $\{1, \dots, \bar S\}$, whatever the forecast; both touches started at the
same size, so $\mathbb E[\Delta S^b - \Delta S^a]$, and with it $\mathbb E[\varepsilon]$, is
close to zero.  Here a queue holds three units, a few events' worth.

In [ ]:
for name in REGIMES:
    size = BOOK[name]["bid"] + TOUCHES[BUY][1]
    print(f"{name}: share of paths by the size at the bid touch")
    print(pd.DataFrame({
        f"size {k}": (size == k).mean(axis=0) for k in range(1, DEPTH + 1)
    }, index=pd.Index(BOOK_HORIZONS, name="h (s)")).round(3).T.to_string())
    print()

The absorption therefore lasts as long as the flow takes to spread over a queue.  Below, the
first half of the same event paths, from the same seed, are folded into a book whose queues hold
four times as much.  The imbalance is the same on every path, since under the assumption an
event contributes its pressure whatever the book (Proposition 1.4.8); only the book differs.

In [ ]:
THICK_DEPTH = UniformDepth(12)
THICK_TOUCHES = {BUY: (1000, 6), SELL: (1041, 6)}

THICK_PATHS = BOOK_PATHS // 2

THICK = {}
for name, params in REGIMES.items():
    THICK[name] = continuations(params, STATE[name], THICK_DEPTH, THICK_TOUCHES, THICK_PATHS,
                                np.random.default_rng(71))
    print(name)
    check(THICK[name], THICK_DEPTH)
    assert np.array_equal(THICK[name]["imbalance"], BOOK[name]["imbalance"][:THICK_PATHS])
    print(f"  the same event paths as the first {THICK_PATHS:,} on the thin book")
    print(book_table(name, THICK[name], STATE[name], THICK_DEPTH)
          .join(dependence(THICK[name], THICK_DEPTH)).round(4).to_string())
    print()

In [ ]:
residual_figure(THICK, STATE, THICK_DEPTH,
                f"From m + e_MB, Sbar = {THICK_DEPTH}: the price inherits the flow late")

On the thicker book almost no quote moves over the first tenth of a second, so
$\mathbb E[\Delta P^m]$ is close to zero there while the expected flow builds up, and
$\mathbb E[\varepsilon]$ is close to minus the flow term.  The mid-price catches up only as the
quotes move: within about a second in the trending specification, and two to three in the
example.  The forecast of the flow is the same on both books, up to the factor $1/(2\bar S)$;
what the book decides is how late the mid-price inherits it.

## 8. The residual bounds the forecast

Remark 1.4.15, the second reason.  At $\rho < 1$, $\Phi(h) \to -K^{-1}$ as $h \to \infty$, and
the flow term tends to $\frac{\tau\bar q}{2\bar S}\varpi^\top\Gamma(I-\Gamma)^{-1}z$, while
$|\varepsilon| \le \tau$ at every horizon.  From $z = m + k\,\mathbf e_{\mathrm{MB}}$ the $m$
part vanishes and the limit is $k\,\theta_{\mathrm{MB}}(\infty)\,\tau\bar q/(2\bar S)$.  On the
book of section 7, with $\bar q = 1$ and $\bar S = 3$, it reaches a tick at
$k = 2\bar S/|\theta_{\mathrm{MB}}(\infty)|$, a burst of market buys.

In [ ]:
BURST = {}
for name, params in REGIMES.items():
    gamma = params.branching_matrix
    progeny = gamma @ np.linalg.inv(IDENTITY - gamma)
    limit = (PRESSURE @ progeny)[EventType.MARKET_BUY]
    BURST[name] = int(np.ceil(2 * DEPTH / abs(limit)))
    print(f"{name:<9} max |Phi(60 s) + K^-1| = "
          f"{np.abs(params.mean_response_integral(Horizon(60.0)) + np.linalg.inv(params.hurwitz_matrix)).max():.1e}   "
          f"theta_MB(inf) = {limit:.4f}   burst k = {BURST[name]}")
    for k in (1, BURST[name]):
        print(f"{'':<9} k = {k}: limit of the flow term "
              f"{PRESSURE @ progeny @ (MEAN_STATE[name] + k * MARKET_BUY) / (2 * DEPTH):+.3f} ticks")

In [ ]:
BURSTS = {
    name: continuations(params, MEAN_STATE[name] + BURST[name] * MARKET_BUY, DEPTH, TOUCHES,
                        4_000, np.random.default_rng(81))
    for name, params in REGIMES.items()
}
for name, run in BURSTS.items():
    print(f"{name}, k = {BURST[name]}")
    check(run, DEPTH)
    print(book_table(name, run, MEAN_STATE[name] + BURST[name] * MARKET_BUY, DEPTH)
          .round(4).to_string())
    print()

In [ ]:
LONG = np.linspace(0.0, 3.0, 61)
figure = go.Figure(go.Scatter(
    x=np.r_[LONG, LONG[::-1]], y=np.r_[np.ones_like(LONG), -np.ones_like(LONG)], fill="toself",
    fillcolor=MUTED, opacity=0.15, line=dict(width=0), name="|ε| ≤ τ", hoverinfo="skip",
))
for name, params in REGIMES.items():
    for k, dash in ((1, "dot"), (BURST[name], "solid")):
        figure.add_trace(go.Scatter(
            x=LONG, y=flow_forecast(params, MEAN_STATE[name] + k * MARKET_BUY, LONG) / (2 * DEPTH),
            mode="lines", name=f"{name}, k = {k}, flow term", line=dict(color=COLOUR[name], dash=dash, width=2),
        ))
    run = BURSTS[name]
    figure.add_trace(go.Scatter(
        x=BOOK_HORIZONS, y=run["mid"].mean(axis=0), mode="markers",
        name=f"{name}, k = {BURST[name]}, E[ΔPᵐ] Monte Carlo",
        error_y=dict(array=2 * run["mid"].std(axis=0, ddof=1) / np.sqrt(run["mid"].shape[0]),
                     thickness=1.5, width=3),
        marker=dict(color=COLOUR[name], size=8, symbol="diamond"),
    ))
figure.update_xaxes(title_text="horizon h (s)")
figure.update_yaxes(title_text="ticks")
figure.update_layout(title="The flow term against the residual's bound, from m + k e_MB",
                     height=480)
figure

From one extra market buy the flow term settles well inside the band at every horizon: a long
horizon does not buy a forecast that exceeds the residual.  From the burst it reaches the edge
of the band, and the Monte Carlo mid-price follows it there.  That is the scale at which
(1.4.14) says something about the mid-price, and it is why section 1.5 asks about the sign of
the forecast rather than its magnitude.

## 9. The contrast is the drift of the imbalance

Corollary 1.4.16: $\Phi(h) = hI + \tfrac12 h^2K + O(h^3)$, so the forecast is
$\bar q\,\Delta\lambda(t+)\,h + O(h^2)$, with $\Delta\lambda(t+) = \varpi^\top A Z(t+)$.  From
$z = m + \mathbf e_{\mathrm{MB}}$ the contrast is $\theta_{\mathrm{MB}}$.  We set $\bar q = 1$,
a factor common to both sides; the gap between the forecast and its tangent should then be
$\tfrac12 h^2\,\varpi^\top A K z$ to leading order.

In [ ]:
SHORT = np.geomspace(1e-3, 1.0, 31)
NEAR = np.linspace(0.0, 0.5, 51)
figure = make_subplots(rows=1, cols=2, subplot_titles=(
    "forecast (solid) and tangent (dashed)", "|forecast - tangent| (solid) and its h² term (dotted)"))
for name, params in REGIMES.items():
    state = STATE[name]
    contrast = PRESSURE @ params.excitation @ state
    curvature = PRESSURE @ params.excitation @ params.hurwitz_matrix @ state / 2
    gap = np.abs(flow_forecast(params, state, SHORT) - contrast * SHORT)
    slope = np.polyfit(np.log(SHORT[:8]), np.log(gap[:8]), 1)[0]
    print(f"{name:<9} contrast at t+ = {contrast:+.4f}   theta_MB = "
          f"{params.signed_excitation(PRESSURE)[EventType.MARKET_BUY]:+.4f}   "
          f"slope of log gap against log h below 5 ms: {slope:.3f}")
    colour = COLOUR[name]
    for x, y, dash, column, shown_in_legend in (
        (NEAR, flow_forecast(params, state, NEAR), "solid", 1, True),
        (NEAR, contrast * NEAR, "dash", 1, False),
        (SHORT, gap, "solid", 2, False),
        (SHORT, abs(curvature) * SHORT ** 2, "dot", 2, False),
    ):
        figure.add_trace(go.Scatter(x=x, y=y, mode="lines", name=name, legendgroup=name,
                                    showlegend=shown_in_legend,
                                    line=dict(color=colour, width=2, dash=dash)), row=1, col=column)
figure.update_xaxes(title_text="horizon h (s)")
figure.update_xaxes(type="log", row=1, col=2)
figure.update_yaxes(type="log", exponentformat="power", row=1, col=2)
figure.update_yaxes(title_text="signed events", row=1, col=1)
figure.update_layout(title="Corollary 1.4.16 from m + e_MB, unit sizes", height=460,
                     margin=dict(t=140), legend=dict(orientation="h", yanchor="bottom", y=1.12, x=0))
figure

The tangent leaves the forecast quadratically, with the coefficient the expansion predicts:
the gap and its $h^2$ term coincide at short horizons.  Up to the mean size, the contrast is
the drift of the imbalance, and a statement about the flow; the price inherits it only through
(1.4.14), residual included.

## 10. The total response

Remark 1.4.17.  At $\rho < 1$, $A\Phi(\infty) = -AK^{-1} = \Gamma(I-\Gamma)^{-1}$, the whole
expected progeny of the present state.  The right Perron vector $u$ of $\Gamma$ is fixed by
$\Sigma$, so $\varpi^\top A u = 0$, and the slowest mode, which decays at the relaxation rate
$\beta(1-\rho)$ of Definition 1.3.35, is absent from the forecast.

In [ ]:
for name, params in REGIMES.items():
    gamma = params.branching_matrix
    progeny = gamma @ np.linalg.inv(IDENTITY - gamma)
    values, vectors = np.linalg.eig(gamma)
    perron = vectors[:, np.argmax(values.real)].real
    perron = perron / perron.sum()
    print(name)
    print(f"  max |-A K^-1 - Gamma (I - Gamma)^-1|  = "
          f"{np.abs(-params.excitation @ np.linalg.inv(params.hurwitz_matrix) - progeny).max():.1e}")
    print(f"  max |A Phi(60 s) - Gamma (I - Gamma)^-1| = "
          f"{np.abs(params.excitation @ params.mean_response_integral(Horizon(60.0)) - progeny).max():.1e}")
    print(f"  Perron eigenvalue {values.real.max():.4f}, u = {perron.round(4)}")
    print(f"  max |Sigma u - u| = {np.abs(SWAP @ perron - perron).max():.1e}   "
          f"varpi' A u = {PRESSURE @ params.excitation @ perron:.1e}")
    print(f"  relaxation rate beta (1 - rho) = {1 / params.relaxation_time():.4f} 1/s")

Which modes survive the contraction with $\varpi$ can be read off $\Sigma$.  A
$\Sigma$-invariant matrix maps the vectors that $\Sigma$ fixes to vectors it fixes, and the
vectors it negates to vectors it negates; in the basis $(\mathbf e_a \pm \mathbf e_{\Sigma a})/\sqrt2$
it is block diagonal.  $\varpi$ lies in the second block and $u$ in the first, so only the
eigenvalues of the second block can reach the forecast.

In [ ]:
FIXED = np.array([UNIT[a] + UNIT[a + 1] for a in (0, 2, 4)]) / np.sqrt(2)
NEGATED = np.array([UNIT[a] - UNIT[a + 1] for a in (0, 2, 4)]) / np.sqrt(2)
BASIS = np.vstack([FIXED, NEGATED]).T

print("varpi in the basis:", (BASIS.T @ PRESSURE).round(4))
for name, params in REGIMES.items():
    blocks = BASIS.T @ params.branching_matrix @ BASIS
    print(name)
    print(f"  largest entry off the two diagonal blocks: "
          f"{max(np.abs(blocks[:3, 3:]).max(), np.abs(blocks[3:, :3]).max()):.1e}")
    for label, block in (("fixed by Sigma  ", blocks[:3, :3]), ("negated by Sigma", blocks[3:, 3:])):
        eigenvalues = np.sort(np.linalg.eigvals(block).real)[::-1]
        print(f"  {label}: eigenvalues of Gamma {eigenvalues.round(4)}, "
              f"rates beta (1 - gamma) {(params.decay * (1 - eigenvalues)).round(3)} 1/s")

The Perron eigenvalue sits in the first block in both regimes.  Within the second, the rate a
pair decays at is that of the slowest mode it actually carries, which the curves themselves
give: $\theta_a(\infty) - \theta_a(h)$ is a sum of the surviving modes, and the slope of its
logarithm at long horizons is the slowest rate present.

In [ ]:
LATE = np.linspace(0.0, 4.0, 81)
rows = []
figure = go.Figure()
for name, params in REGIMES.items():
    gamma = params.branching_matrix
    limit = (PRESSURE @ gamma @ np.linalg.inv(IDENTITY - gamma)) / PRESSURE
    remaining = np.array([limit - params.signed_excitation_at_horizon(PRESSURE, Horizon(h))
                          for h in LATE])
    for pair, event in PAIRS.items():
        if np.abs(remaining[:, event]).max() < 1e-12:
            rows.append({"regime": name, "pair": pair, "theta(inf)": limit[event],
                         "decay rate (1/s)": "theta = 0 at every h"})
            continue
        last = LATE >= 3.0
        rate = -np.polyfit(LATE[last], np.log(np.abs(remaining[last, event])), 1)[0]
        rows.append({"regime": name, "pair": pair, "theta(inf)": limit[event],
                     "decay rate (1/s)": f"{rate:.3f}"})
        figure.add_trace(go.Scatter(
            x=LATE, y=np.abs(remaining[:, event]), mode="lines", name=f"{name}, {pair}",
            line=dict(color=COLOUR[name], dash=DASH[pair], width=2),
        ))
relaxation = 1 / REGIMES["example"].relaxation_time()
figure.add_trace(go.Scatter(
    x=LATE, y=2 * np.exp(-relaxation * LATE), mode="lines", name="∝ exp(-β(1-ρ)h)",
    line=dict(color=MUTED, width=1.5, dash="dashdot"),
))
print(pd.DataFrame(rows).round(4).to_string(index=False))
figure.update_xaxes(title_text="horizon h (s)")
figure.update_yaxes(type="log", title_text="|θ(∞) - θ(h)|", exponentformat="power")
figure.update_layout(title="What remains of the forecast, pair by pair, against the relaxation rate",
                     height=460, legend=dict(orientation="h", y=1.16, x=0))
figure

Every pair finishes faster than the relaxation rate.  In the example the market pair decays at
the rate of the second block's leading eigenvalue, and the withdrawal pair at $\beta$ itself.
What a withdrawal begets is limit orders, whose $\theta(h)$ is zero at every $h$, and mirror
pairs in equal numbers, whose signed forecasts cancel; so its signed forecast is its direct
offspring and nothing more, $\theta_W(h) = \theta_W(1 - e^{-\beta h})/\beta$.  In the trending
specification every pair decays at the second block's leading rate, close to the relaxation rate.

In [ ]:
params = REGIMES["example"]
withdrawal = params.signed_excitation(PRESSURE)[EventType.WITHDRAW_BUY]
direct = withdrawal * (1 - np.exp(-params.decay * LATE)) / params.decay
print("example: max |theta_W(h) - theta_W (1 - exp(-beta h)) / beta| =",
      np.abs([params.signed_excitation_at_horizon(PRESSURE, Horizon(h))[EventType.WITHDRAW_BUY]
              for h in LATE] - direct).max())

## 11. The sign belongs to the kernel

Corollary 1.4.18: $\theta(h)$ of (1.4.16) is constant on pairs, $\theta(h)/h \to \theta$ as
$h \to 0$, and an extra event of a type $a$ with $\varpi_a = +1$ against one fewer of its
mirror moves the forecast by $2\bar q\,\theta_a(h)$.

In [ ]:
CURVE = np.linspace(0.0, 3.0, 61)[1:]
figure = go.Figure()
for name, params in REGIMES.items():
    curves = np.array([params.signed_excitation_at_horizon(PRESSURE, Horizon(h)) for h in CURVE])
    assert np.allclose(curves[:, 0::2], curves[:, 1::2])
    small = params.signed_excitation_at_horizon(PRESSURE, Horizon(1e-4)) / 1e-4
    print(f"{name:<9} theta(h)/h at h = 0.1 ms: {small[0::2].round(3)}   "
          f"theta: {params.signed_excitation(PRESSURE)[0::2].round(3)}")
    for pair, event in PAIRS.items():
        figure.add_trace(go.Scatter(x=CURVE, y=curves[:, event], mode="lines", name=f"{name}, {pair}",
                                    line=dict(color=COLOUR[name], dash=DASH[pair], width=2)))
figure.update_xaxes(title_text="horizon h (s)")
figure.update_yaxes(title_text="θ(h)")
figure.update_layout(title="Signed excitation at a horizon: opposite signs in the two regimes",
                     height=440, legend=dict(orientation="h", y=1.16, x=0))
figure

The vector $m + \mathbf e_{\mathrm{MB}} - \mathbf e_{\mathrm{MS}}$ is not a state.  $Z_e$ is a
sum of decayed ones, so it is non-negative, and the mean state holds less than one market sell;
the simulator refuses it.

In [ ]:
for name in REGIMES:
    print(f"{name:<9} m_MS = {MEAN_STATE[name][EventType.MARKET_SELL]:.3f}")
try:
    ExponentialHawkes.from_state(REGIMES["example"],
                                 MEAN_STATE["example"] + MARKET_BUY - MARKET_SELL,
                                 np.random.default_rng(0))
except ValueError as refusal:
    print(refusal)

The displacement is the difference of two forecasts, so we read it off two fans: the one of
section 6, from $m + \mathbf e_{\mathrm{MB}}$, and a new one from $m + \mathbf e_{\mathrm{MS}}$.
Since $\varpi^\top A\Phi(h)m = 0$, the forecasts are exactly $\pm\bar q\,\theta_{\mathrm{MB}}(h)$,
and their difference is $2\bar q\,\theta_{\mathrm{MB}}(h)$.  The paths of the two fans are
independent, so pairing them in any order gives independent samples of the difference.

In [ ]:
MIRROR_FANS = {
    name: signed_flow(params, MARKS[name], MEAN_STATE[name] + MARKET_SELL, PATHS,
                      np.random.default_rng(62))
    for name, params in REGIMES.items()
}
DISPLACEMENT = {}
for name, params in REGIMES.items():
    theta_h = np.array([params.signed_excitation_at_horizon(PRESSURE, Horizon(h))[EventType.MARKET_BUY]
                        for h in HORIZONS])
    DISPLACEMENT[name] = 2 * MEAN_SIZE[name] * theta_h
    print(name)
    print(pd.concat({
        "from m + e_MB": against(FANS[name][1], MEAN_SIZE[name] * theta_h, HORIZONS),
        "from m + e_MS": against(MIRROR_FANS[name][1], -MEAN_SIZE[name] * theta_h, HORIZONS),
        "difference": against(FANS[name][1] - MIRROR_FANS[name][1], DISPLACEMENT[name], HORIZONS),
    }, axis=1).drop(columns="standard error", level=1).round(2).to_string())
    print()

In [ ]:
monte_carlo_figure(
    HORIZONS,
    np.stack([FANS[name][1] - MIRROR_FANS[name][1] for name in REGIMES], axis=2),
    np.stack([DISPLACEMENT[name] for name in REGIMES], axis=1),
    list(REGIMES),
    "The displacement 2 qbar theta_MB(h): two fans, differenced",
).update_xaxes(title_text="horizon h (s)").update_yaxes(title_text="shares")

Each fan meets its own forecast, and the difference meets $2\bar q\,\theta_{\mathrm{MB}}(h)$,
negative in the example and positive in the trending specification at every horizon: the same
excess of market buys over market sells is forecast to reverse the flow in one specification and
to continue it in the other.  The two share the branching ratio and the total rate, and
the identity of section 7 holds identically in both.

At a general state the forecast is $\bar q\sum_e\varpi_e\theta_e(h)Z_e(t+)$, and two
specifications of opposite sign need not disagree on it.  One such state: a limit buy has just
arrived, on the mean state, and a withdrawal from the bid arrived one half-life of the kernel
before it, so that it carries a weight of one half.

In [ ]:
MIXED = {name: MEAN_STATE[name] + UNIT[EventType.LIMIT_BUY] + 0.5 * UNIT[EventType.WITHDRAW_BUY]
         for name in REGIMES}
print("half-life of the kernel, log 2 / beta:", round(np.log(2) / PATH.decay, 4), "s")
print(pd.DataFrame({
    name: flow_forecast(params, MIXED[name], HORIZONS) for name, params in REGIMES.items()
}, index=pd.Index(HORIZONS, name="h (s)")).round(4).to_string())

Both specifications forecast an upward flow from this state, at every horizon.  In the example
the limit buy begets nothing signed and the withdrawal begets its reversal, which pushes up; in
the trending specification the limit buy's continuation outweighs the half-decayed withdrawal's.
Such states occur along an ordinary path: read after each event of the window of section 4, the
two forecasts share their sign at the events counted below, and oppose it at the others.

In [ ]:
after_each = decayed_counts_at(PATH.decay, times, types, np.zeros(6), times) + UNIT[types]
agree = np.sign(after_each[shown] @ (PRESSURE @ REGIMES["example"].excitation
                                     @ REGIMES["example"].mean_response_integral(Horizon(1.0)))) == \
        np.sign(after_each[shown] @ (PRESSURE @ REGIMES["trending"].excitation
                                     @ REGIMES["trending"].mean_response_integral(Horizon(1.0))))
print(f"at h = 1 s the two forecasts share their sign at {agree.sum()} of the {shown.sum()} "
      f"events of section 4's window, read from Z(T+)")

## 12. Three statistics of the same past flow

Remark 1.4.19, on the path of section 4.  We stand at the time $t$ of the last event of the
window, and give each past event the weight each statistic puts on it: the signed count over
$(t - w, t]$ gives it $\varpi_{E_n}$; the decayed count $\varpi^\top Z$ gives it
$\varpi_{E_n}e^{-\beta(t - T_n)}$; the imbalance over $(t - w, t]$ gives it
$\varpi_{E_n}q_n$; and the forecast gives it $\bar q\,\varpi_{E_n}\theta_{E_n}(h)e^{-\beta(t - T_n)}$.
The event at $t$ lies in the window, so the decayed count and the forecast are read from
$Z(t+)$, where that event has age zero and weight one; $Z(t)$, left-continuous, would leave it
out of both.

In [ ]:
WINDOW = 1.0
FORECAST_HORIZON = Horizon(1.0)
now = times[shown][-1]
past = times <= now
age = now - times[past]
kinds = types[past]
sizes = draw_sizes(MARKS["example"], np.random.default_rng(12), past.sum())
in_window = age < WINDOW

state_after = np.exp(-PATH.decay * age) @ UNIT[kinds]
replayed = decayed_counts_at(PATH.decay, times, types, np.zeros(6), np.array([now]))[0] + UNIT[kinds[-1]]
print(f"t = {now:.4f} s, {in_window.sum()} events in (t - w, t]; "
      f"max |Z(t+) - (Z(t-) + e_E)| = {np.abs(state_after - replayed).max():.1e}")
print()
statistics = {
    "signed count (events)": PRESSURE[kinds][in_window].sum(),
    "decayed count varpi' Z(t+) (events)": PRESSURE @ state_after,
    "imbalance OFI (shares)": (PRESSURE[kinds] * sizes)[in_window].sum(),
}
for name, params in REGIMES.items():
    statistics[f"forecast over 1 s, {name} (shares)"] = MEAN_SIZE[name] * (
        PRESSURE @ params.excitation @ params.mean_response_integral(FORECAST_HORIZON) @ state_after)
print(pd.Series(statistics).round(3).to_string())

In [ ]:
AGES = np.linspace(0.0, 1.5, 301)
figure = make_subplots(rows=1, cols=2, subplot_titles=list(REGIMES), shared_yaxes=True)
for column, (name, params) in enumerate(REGIMES.items(), start=1):
    first = column == 1
    figure.add_trace(go.Scatter(x=AGES, y=(AGES < WINDOW).astype(float), mode="lines",
                                name="signed count", line=dict(color=MUTED, width=2, shape="hv"),
                                legendgroup="count", showlegend=first), row=1, col=column)
    figure.add_trace(go.Scatter(x=AGES, y=np.exp(-params.decay * AGES), mode="lines",
                                name="decayed count", line=dict(color=INK, width=2),
                                legendgroup="decayed", showlegend=first), row=1, col=column)
    figure.add_trace(go.Scatter(x=age[in_window], y=sizes[in_window] / MEAN_SIZE[name], mode="markers",
                                name="imbalance, q/qbar, this path", marker=dict(color=PALETTE[3], size=8),
                                legendgroup="ofi", showlegend=first), row=1, col=column)
    theta_h = params.signed_excitation_at_horizon(PRESSURE, FORECAST_HORIZON)
    for pair, event in PAIRS.items():
        figure.add_trace(go.Scatter(x=AGES, y=theta_h[event] * np.exp(-params.decay * AGES),
                                    mode="lines", name=f"forecast, {name}, {pair}",
                                    line=dict(color=COLOUR[name], dash=DASH[pair], width=2)),
                         row=1, col=column)
figure.update_xaxes(title_text="age of the event, t - Tₙ (s)")
figure.update_yaxes(title_text="weight per unit of pressure", col=1)
figure.update_layout(title="The weight each statistic puts on a past event (sizes in units of qbar)",
                     height=480)
figure

At this $t$ the signed count, the decayed count and the imbalance agree in sign, and the two
forecasts take opposite signs: the example's against the three statistics, the trending one's
with them.

The count and the imbalance weight the window flat and ignore everything older; the decayed
count and the forecast weight by age, on the kernel's own decay.  They differ by type: the
forecast scales the decayed count by $\theta_e(h)$, which in the example is negative on two pairs
and zero on the limit orders, so the events the imbalance counts most often are the ones the
example's forecast ignores.  And only the imbalance sees the sizes.

The gap in the weighting closes when $\theta$ is constant across the six types, that is when
$\varpi$ is a left eigenvector of $A$.  One kernel with this property:
$A_{ee'} = r_e\,a_{\mathrm{within}}$ when $e$ and $e'$ push the same way and
$r_e\,a_{\mathrm{across}}$ when they do not, with $r_e = r_{\Sigma e}$.  The rows of either
pressure then carry the same total $r$, so
$(\varpi^\top A)_{e'} = \varpi_{e'}(a_{\mathrm{within}} - a_{\mathrm{across}})
\sum_{e : \varpi_e = +1} r_e$.  We take $r = \lambda^*$ of the example, which makes $\lambda^*$
the Perron vector and $\mu = (1 - \rho)\lambda^*$ positive, and build it at the example's
branching ratio and stationary intensity.

In [ ]:
target = PATH.stationary_intensity()
constant_theta = HawkesParams.from_stationary_intensity(
    target, target[:, None] * np.where(np.equal.outer(PRESSURE, PRESSURE), 2.0, 1.0),
    PATH.decay, BranchingRatio(PATH.branching_ratio))
row = PRESSURE @ constant_theta.excitation
print("theta:", constant_theta.signed_excitation(PRESSURE).round(4))
print("direction-symmetric:",
      np.allclose(SWAP @ constant_theta.excitation @ SWAP, constant_theta.excitation)
      and np.allclose(SWAP @ constant_theta.baseline, constant_theta.baseline))
print(f"max |varpi' A - theta varpi'| = {np.abs(row - row[0] * PRESSURE).max():.1e}")
for h in (0.25, 1.0, 3.0):
    weights = PRESSURE @ constant_theta.excitation @ constant_theta.mean_response_integral(Horizon(h))
    print(f"h = {h}: max |varpi' A Phi(h) - theta(h) varpi'| = "
          f"{np.abs(weights - weights[0] * PRESSURE).max():.1e}")

weights = PRESSURE @ constant_theta.excitation @ constant_theta.mean_response_integral(FORECAST_HORIZON)
forecast_constant = weights @ state_after
print(f"forecast per unit of mean size {forecast_constant:.4f}   "
      f"theta(h) x decayed count {weights[0] * (PRESSURE @ state_after):.4f}")

With $\theta$ constant the forecast is a multiple of the decayed count at every horizon.  The
gap in the marks does not close.  Redrawing the sizes of the same events leaves the state, and
every function of it, where it was, and moves the imbalance.

In [ ]:
for seed in (13, 14, 15):
    redrawn = draw_sizes(MARKS["example"], np.random.default_rng(seed), past.sum())
    print(f"sizes redrawn: OFI = {(PRESSURE[kinds] * redrawn)[in_window].sum():6.0f} shares   "
          f"decayed count = {PRESSURE @ state_after:.4f}   forecast = {forecast_constant:.4f}")

## What to take away

The forecast of the flow is a function of the state read after the event, and direction
symmetry removes every baseline term from it.  The Monte Carlo meets it in both regimes, and
tells it apart from the forecast read before the event.

The mid-price inherits the forecast through the identity, residual included.  The residual
absorbs the flow while it goes into the queues, briefly on a thin book and for longer on a thick
one, and it bounds what the forecast can say about the mid-price: only a state loaded far enough
moves it by a tick.

The sign belongs to the kernel.  From the same event, the example forecasts reversal and the
trending specification continuation, at the same branching ratio and total rate; at a general
state they need not disagree.

The signed count, the decayed count and the imbalance weight the same past events three ways,
and the forecast weights them a fourth way.  The imbalance is the only one of the four that sees
the sizes.